# Step4: Signature and Target Analysis

Project-specific downstream notebook for treatment-response modules, pathway scoring, target-focused analysis, and export matrices.

This notebook is intentionally more project-specific than Step3. Keep reusable analysis policy in docs and stable APIs; use this notebook to preserve real exploratory decisions and export logic.

Input: `data/processed/Step3-sce_annotated.h5ad`.


---

## 1. Setup

### 1.1 Imports


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import scLucid as scl
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
import warnings
warnings.filterwarnings('ignore')

scl.setup_logging("INFO")
scl.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 5),
    style="seaborn-v0_8",
    style_dict={'axes.grid': True},
    color_theme="default"
)


### 1.2 Paths


In [ ]:
# Define working directories
BASE_DIR = '/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/Ongoing/202604JJH'
DATA_DIR = os.path.join(BASE_DIR, "1-DATA")
RESULTS_DIR = os.path.join(BASE_DIR, "2-OUTPUT")
os.makedirs(RESULTS_DIR, exist_ok=True)


### 1.3 Colors


In [ ]:
###** colors
color = {
    'PBS': '#BFCFE3',
    "R301":"#8B4C9D"
}


In [ ]:
print("Loading annotated data...")
adata = sc.read_h5ad(os.path.join(DATA_DIR, "Step3-sce_annotated.h5ad"))
print(f"Dataset loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")
if "celltype" not in adata.obs.columns:
    raise KeyError("Expected adata.obs['celltype']; run Step2-Annotation_and_Malignancy.ipynb first.")


In [ ]:
# ==============================================================================
# Malignant-only pathway / module scoring
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

score_fig_dir = os.path.join(RESULTS_DIR, "figures", "malignant_module_scores")
os.makedirs(score_fig_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Define malignant cell types
# ------------------------------------------------------------------------------

malignant_celltypes = [
    "Mesenchymal-like malignant cells",
    "Growth factor-high malignant cells",
    "Translation-high malignant cells",
    "Hypoxic/glycolytic malignant cells",
    "Cycling malignant cells",
    "IFN-stimulated malignant cells",
    "Proliferating malignant cells",
    "CDH13+ mesenchymal-like malignant cells"
]

malignant_celltypes = [
    ct for ct in malignant_celltypes
    if ct in adata.obs["celltype"].unique()
]

adata_tumor = adata[adata.obs["celltype"].isin(malignant_celltypes)].copy()

print(adata_tumor)
print(adata_tumor.obs["celltype"].value_counts())

In [ ]:
# ==============================================================================
# 2. Define module gene sets
# ==============================================================================

module_gene_sets = {
    # Proliferation / cell cycle
    "Proliferation": [
        "Mki67", "Top2a", "Prc1", "Ube2c", "Cenpf", "Cenpe",
        "Pbk", "Nusap1", "Kif11", "Kif20a", "Kif23", "Hmmr"
    ],

    "G2M_cell_cycle": [
        "Ccnb1", "Ccnb2", "Cdc20", "Cdc25c", "Plk1", "Aurka",
        "Aurkb", "Tpx2", "Cenpa", "Cenpe", "Cenpf", "Kif2c"
    ],

    "DNA_replication": [
        "Mcm2", "Mcm3", "Mcm4", "Mcm5", "Mcm6", "Mcm7",
        "Pcna", "Rrm1", "Rrm2", "Tyms", "Fen1", "Ung"
    ],

    # Translation / ribosome
    "Translation": [
        "Eif4a1", "Eif3a", "Eif3b", "Eif3e", "Eif4e",
        "Eef1a1", "Eef1b2", "Eef2", "Rplp0", "Rps3", "Rps6"
    ],

    "Ribosome": [
        "Rpl3", "Rpl4", "Rpl5", "Rpl7", "Rpl8", "Rpl10",
        "Rpl13", "Rpl13a", "Rplp0", "Rps3", "Rps6", "Rps8"
    ],

    # Hypoxia / metabolism
    "Hypoxia": [
        "P4ha1", "P4ha2", "Plod2", "Ankrd37", "Bnip3", "Bnip3l",
        "Ndrg1", "Pdk1", "Higd1a", "Vegfa", "Slc2a1", "Ldha"
    ],

    "Glycolysis": [
        "Hk1", "Hk2", "Gpi1", "Pfkl", "Pfkp", "Aldoa",
        "Gapdh", "Pgk1", "Pgam1", "Eno1", "Eno2", "Pkm", "Ldha"
    ],

    "OXPHOS": [
        "Ndufa1", "Ndufa2", "Ndufa4", "Ndufb5", "Ndufs1",
        "Sdha", "Uqcrc1", "Cox4i1", "Cox5a", "Atp5f1a", "Atp5f1b"
    ],

    # IFN / inflammation
    "IFN_response": [
        "Ifit1", "Ifit2", "Ifit3", "Ifit3b", "Isg15", "Usp18",
        "Oasl2", "Oas1a", "Oas2", "Rsad2", "Ifih1", "Mx1", "Stat1"
    ],

    "Inflammatory_response": [
        "Il1b", "Il6", "Tnf", "Cxcl1", "Cxcl2", "Cxcl3",
        "Ccl2", "Ccl7", "Ptgs2", "Nfkbia", "Socs3"
    ],

    # EMT / mesenchymal-like
    "EMT_mesenchymal": [
        "Vim", "Fn1", "Col1a1", "Col1a2", "Col3a1", "Col5a2",
        "Prrx1", "Zeb1", "Zeb2", "Snai1", "Snai2", "Twist1",
        "Cdh13", "Sema3a", "Slit2"
    ],

    "ECM_remodeling": [
        "Col1a1", "Col1a2", "Col3a1", "Col5a2", "Col6a2",
        "Col6a3", "Mmp2", "Mmp9", "Mmp14", "Lox", "Loxl2", "Fbn1"
    ],

    # Growth factor / receptor signaling
    "Growth_factor_signaling": [
        "Ereg", "Hbegf", "Areg", "Btc", "Epgn", "Grem1",
        "Timp1", "Egfr", "Erbb2", "Met", "Igf1r", "Itga6"
    ],

    "MAPK_EGFR_related": [
        "Ereg", "Hbegf", "Areg", "Egfr", "Dusp1", "Dusp4",
        "Dusp6", "Fos", "Jun", "Egr1", "Spry2", "Spry4"
    ],

    # Antigen presentation / immune visibility
    "MHC_I_antigen_presentation": [
        "B2m", "H2-K1", "H2-D1", "Tap1", "Tap2", "Tapbp",
        "Psmb8", "Psmb9", "Psmb10", "Nlrc5"
    ],

    "MHC_II_antigen_presentation": [
        "H2-Aa", "H2-Ab1", "H2-Eb1", "H2-Ea", "Cd74",
        "Ciita", "H2-DMa", "H2-DMb1", "H2-DMb2"
    ],

    # Stress / death
    "Apoptosis": [
        "Bax", "Bak1", "Bbc3", "Pmaip1", "Casp3", "Casp7",
        "Casp8", "Casp9", "Fas", "Fasl", "Tnfrsf10b"
    ],

    "p53_pathway": [
        "Trp53", "Cdkn1a", "Mdm2", "Bax", "Bbc3", "Pmaip1",
        "Gadd45a", "Sesn1", "Sesn2", "Rrm2b"
    ],

    "DNA_damage_response": [
        "Atm", "Atr", "Chek1", "Chek2", "Brca1", "Brca2",
        "Rad51", "H2afx", "Gadd45a", "Ddb2", "Xrcc5"
    ],

    # Tumor-neutrophil axis
    "Neutrophil_recruitment": [
        "Cxcl1", "Cxcl2", "Cxcl3", "Cxcl5", "Csf3",
        "Il1b", "Ccl2", "Ccl7", "Lcn2", "S100a8", "S100a9"
    ]
}

In [ ]:
# ==============================================================================
# 3. Score modules in malignant cells
# ==============================================================================

def filter_genes(gene_list, adata_obj):
    """Keep genes present in adata.var_names."""
    return [g for g in gene_list if g in adata_obj.var_names]

score_names = []

for module_name, genes in module_gene_sets.items():
    genes_use = filter_genes(genes, adata_tumor)
    
    if len(genes_use) < 3:
        print(f"Skip {module_name}: only {len(genes_use)} genes found.")
        continue
    
    score_name = module_name + "_score"
    
    sc.tl.score_genes(
        adata_tumor,
        gene_list=genes_use,
        score_name=score_name,
        ctrl_size=min(50, len(genes_use) * 2),
        random_state=0,
        use_raw=True
    )
    
    score_names.append(score_name)
    print(f"{module_name}: {len(genes_use)} genes used.")

print("\nScores added:")
print(score_names)

In [ ]:
# ==============================================================================
# 4. Summarize module scores by group and malignant state
# ==============================================================================

# Mean score by treatment group
score_by_group = (
    adata_tumor.obs
    .groupby("group")[score_names]
    .mean()
)

# Mean score by malignant cell state
score_by_celltype = (
    adata_tumor.obs
    .groupby("celltype")[score_names]
    .mean()
)

# Mean score by group and malignant state
score_by_group_celltype = (
    adata_tumor.obs
    .groupby(["group", "celltype"])[score_names]
    .mean()
)

print("Score by group:")
display(score_by_group.round(3))

print("Score by malignant cell state:")
display(score_by_celltype.round(3))

In [ ]:
score_by_group.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_group.csv"))
score_by_celltype.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_malignant_celltype.csv"))
score_by_group_celltype.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_group_and_malignant_celltype.csv"))

In [ ]:
# ==============================================================================
# 5. Violin plots: PBS vs R301 in malignant cells
# ==============================================================================

selected_scores = [
    "Proliferation_score",
    "G2M_cell_cycle_score",
    "Translation_score",
    "Hypoxia_score",
    "Glycolysis_score",
    "IFN_response_score",
    "EMT_mesenchymal_score",
    "Growth_factor_signaling_score",
    "MHC_I_antigen_presentation_score",
    "Neutrophil_recruitment_score"
]

selected_scores = [s for s in selected_scores if s in adata_tumor.obs.columns]

sc.pl.violin(
    adata_tumor,
    keys=selected_scores,
    groupby="group",
    rotation=45,
    stripplot=False,
    multi_panel=True,
    show=False
)

plt.savefig(
    os.path.join(score_fig_dir, "Violin_ModuleScores_by_group_malignant_only.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# 6. Dotplot: module scores by malignant cell state
# ==============================================================================

# Scanpy dotplot一般用于基因表达。
# 对obs里的score，更方便用matrixplot-like heatmap，见下一段。
# 这里先画 UMAP feature plot。

for score in selected_scores:
    sc.pl.umap(
        adata_tumor,
        color=score,
        cmap="viridis",
        show=False
    )
    
    plt.savefig(
        os.path.join(score_fig_dir, f"UMAP_{score}.pdf"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
# ==============================================================================
# 8. Heatmap: ALL computed module scores across treatment groups
# ==============================================================================

# Use ALL module score columns (not just selected_scores subset)
all_module_score_cols = [c for c in adata_tumor.obs.columns if c.endswith('_score')]
# Exclude generic QC scores, keep only pathway modules
exclude_scores = ['S_score', 'G2M_score', 'doubletdetection_score', 'heuristic_confidence_score', 'cnv_score']
all_module_score_cols = [c for c in all_module_score_cols if c not in exclude_scores]
all_module_score_cols = sorted(all_module_score_cols)

print(f"Plotting {len(all_module_score_cols)} module scores: {all_module_score_cols}")

group_heat_df = (
    adata_tumor.obs
    .groupby("group")[all_module_score_cols]
    .mean()
)

# Reorder groups if treatment_order exists
if "treatment_order" in globals():
    group_heat_df = group_heat_df.reindex([g for g in treatment_order if g in group_heat_df.index])

group_heat_df_z = (group_heat_df - group_heat_df.mean(axis=0)) / group_heat_df.std(axis=0)
group_heat_df_z = group_heat_df_z.fillna(0)

# Larger figure for all modules
fig, ax = plt.subplots(figsize=(14, 3))

im = ax.imshow(group_heat_df_z.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)

ax.set_xticks(np.arange(len(group_heat_df_z.columns)))
ax.set_xticklabels(
    [c.replace("_score", "").replace("_", " ") for c in group_heat_df_z.columns],
    rotation=45,
    ha="right",
    fontsize=8
)

ax.set_yticks(np.arange(len(group_heat_df_z.index)))
ax.set_yticklabels(group_heat_df_z.index, fontsize=10)

ax.set_title("All Pathway Module Changes in Malignant Cells (R301 vs PBS)", fontsize=13, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("Z-scored mean module score")

plt.tight_layout()

plt.savefig(
    os.path.join(score_fig_dir, "Heatmap_ALL_ModuleScores_by_group_malignant.pdf"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Also save the raw mean values for reference
group_heat_df.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_group_raw_values.csv"))
print(f"Saved comprehensive heatmap to: {score_fig_dir}")



## 7. Advanced Treatment-Response Analyses

针对三抗（R301）治疗进行免疫微环境重塑、跨细胞类型联合分析和功能模块全 compartments 比较。

### 7.1 Immune Infiltration & TME Remodeling (Sample-Level)

计算每个样本的免疫浸润分数、基质分数及其亚组分（T细胞、NK、髓系、细胞毒性），
在样本水平比较 PBS vs R301。

In [ ]:
# ==============================================================================
# 7.1 Immune Infiltration & TME Remodeling (Sample-Level)
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

tme_dir = os.path.join(RESULTS_DIR, 'tme_remodeling')
os.makedirs(tme_dir, exist_ok=True)

# --------------------------------------------------------------------------
# Define gene signatures (mouse)
# --------------------------------------------------------------------------
signatures = {
    'Immune_Score': [
        'Ptprc', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Cd19', 'Cd79a', 'Cd79b',
        'Ncr1', 'Nkg7', 'Cd14', 'Cd68', 'Itgam', 'Itgax', 'Fcgr3',
        'Csf1r', 'Cxcr3', 'Ccr7', 'Il2rb', 'Gzmb', 'Prf1'
    ],
    'Stromal_Score': [
        'Col1a1', 'Col1a2', 'Col3a1', 'Col5a2', 'Col6a2', 'Fbn1', 'Dcn', 'Lum',
        'Mmp2', 'Timp1', 'Acta2', 'Tagln', 'Pecam1', 'Cdh5', 'Eng', 'Vwf',
        'Rgs5', 'Pdgfrb', 'Cspg4'
    ],
    'T_Cell_Score': [
        'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Cd2', 'Cd28', 'Trac', 'Trbc1', 'Trbc2'
    ],
    'NK_Cell_Score': [
        'Ncr1', 'Nkg7', 'Klrb1c', 'Klrd1', 'Prf1', 'Gzmb', 'Ctsw', 'Txk'
    ],
    'Myeloid_Score': [
        'Cd14', 'Cd68', 'Itgam', 'Csf1r', 'Fcgr1', 'Fcgr3', 'Adgre1', 'Mertk', 'Trem2', 'Aif1'
    ],
    'Cytotoxic_Score': [
        'Gzma', 'Gzmb', 'Gzmk', 'Prf1', 'Ctsw', 'Nkg7', 'Ifng'
    ],
}

# Score each cell using adata.raw
for sig_name, genes in signatures.items():
    genes_avail = [g for g in genes if g in adata.raw.var_names]
    if len(genes_avail) >= 3:
        sc.tl.score_genes(
            adata, gene_list=genes_avail, score_name=sig_name,
            use_raw=True, ctrl_size=min(50, len(genes_avail) * 2), random_state=42
        )
        print(f'Scored {sig_name}: {len(genes_avail)}/{len(genes)} genes')

score_cols = [s for s in signatures.keys() if s in adata.obs.columns]
print(f'\nTotal score columns added: {len(score_cols)}')

In [ ]:
# --------------------------------------------------------------------------
# Aggregate to sample level & visualize
# --------------------------------------------------------------------------

# Per-sample mean scores
sample_scores = adata.obs.groupby('sampleID')[score_cols].mean()
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
sample_scores = sample_scores.join(sample_to_group)
sample_scores = sample_scores.sort_values('group')

existing_groups = [g for g in ['PBS', 'R301'] if g in sample_scores['group'].unique()]

# ---- A) Paired boxplots with MWU p-values ----
n_cols = 3
n_rows = int(np.ceil(len(score_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

mw_results = []
for i, sc_name in enumerate(score_cols):
    ax = axes[i]
    # Boxplot
    sns.boxplot(data=sample_scores, x='group', y=sc_name, order=existing_groups,
                palette=color, width=0.5, ax=ax)
    sns.stripplot(data=sample_scores, x='group', y=sc_name, order=existing_groups,
                   color='black', size=7, jitter=0.12, alpha=0.85, ax=ax)
    
    # MWU test
    a = sample_scores.loc[sample_scores['group'] == 'R301', sc_name].values
    b = sample_scores.loc[sample_scores['group'] == 'PBS', sc_name].values
    if len(a) >= 2 and len(b) >= 2:
        u, p = mannwhitneyu(a, b, alternative='two-sided')
        mw_results.append({'score': sc_name, 'u_stat': u, 'p_value': p})
    else:
        mw_results.append({'score': sc_name, 'u_stat': np.nan, 'p_value': np.nan})
    
    ax.set_title(sc_name.replace('_', ' '), fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Mean Score per Sample')

# Hide unused axes
for j in range(len(score_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(tme_dir, 'TME_Scores_boxplot.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# FDR correction
mw_df = pd.DataFrame(mw_results)
mw_df['p_fdr'] = multipletests(mw_df['p_value'].fillna(1.0).values, method='fdr_bh')[1]
mw_df.to_csv(os.path.join(tme_dir, 'TME_Scores_MWU.csv'), index=False)
print('\nTME Score Comparison (R301 vs PBS, MWU + FDR):')
print(mw_df.round(4).to_string(index=False))

In [ ]:
# ---- B) Immune-Stromal balance barplot per sample ----
fig, ax = plt.subplots(figsize=(9, 5))

plot_df = sample_scores[['Immune_Score', 'Stromal_Score']].copy()
plot_df = plot_df.join(sample_to_group)
plot_df = plot_df.sort_values('group')

x = range(len(plot_df))
width = 0.35
labels = [f'{r.name}\n({r["group"]})' for _, r in plot_df.iterrows()]

ax.bar([i - width/2 for i in x], plot_df['Immune_Score'], width,
       label='Immune Score', color='#3498db', alpha=0.85)
ax.bar([i + width/2 for i in x], plot_df['Stromal_Score'], width,
       label='Stromal Score', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean Score per Sample')
ax.set_title('Immune vs Stromal Score by Sample', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', frameon=False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(tme_dir, 'Immune_Stromal_balance_barplot.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ---- C) Heatmap: Z-scored scores across samples ----
z_df = (sample_scores[score_cols] - sample_scores[score_cols].mean()) / sample_scores[score_cols].std()
z_df = z_df.fillna(0)

fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.imshow(z_df.T.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(z_df)))
ax.set_xticklabels([f'{r.name} ({r["group"]})' for _, r in plot_df.iterrows()],
                   rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(score_cols)))
ax.set_yticklabels([s.replace('_Score', '') for s in score_cols], fontsize=9)
ax.set_title('Z-scored TME Signatures by Sample', fontsize=13, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Z-score')
plt.tight_layout()
plt.savefig(os.path.join(tme_dir, 'TME_Scores_heatmap.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f'TME remodeling figures saved to: {tme_dir}')

### 7.2 Cross-Celltype Shared DEG Analysis

从每种细胞类型的 DEG 结果中，找出在多种细胞类型中共同上调或下调的基因。
这些跨细胞类型的共享 DEG 可能代表三抗治疗的全局性转录重编程效应。

In [ ]:
# ==============================================================================
# 7.2 Cross-Celltype Shared DEG Analysis
# ==============================================================================

import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

shared_dir = os.path.join(RESULTS_DIR, 'shared_degs')
os.makedirs(shared_dir, exist_ok=True)

# --------------------------------------------------------------------------
# 1) Load all per-celltype significant DEGs
# --------------------------------------------------------------------------
DEG_DIR = os.path.join(BASE_DIR, 'DEG_Enrichr_all_comparisons/R301_vs_PBS')
sig_files = sorted(glob.glob(os.path.join(DEG_DIR, '*_DEG_sig.csv')))
print(f'Found {len(sig_files)} significant DEG files')

LFC_THRESHOLD = 0.5
PVAL_THRESHOLD = 0.05

# Parse celltype from filename
def parse_celltype(filename):
    base = os.path.basename(filename)
    # Remove _DEG_sig.csv suffix
    ct = base.replace('_DEG_sig.csv', '').replace('_', ' ').replace('  ', ' ')
    return ct

# Collect all DEGs
deg_by_celltype = {}
for f in sig_files:
    ct = parse_celltype(f)
    df = pd.read_csv(f)
    # Standardize column names
    if 'logfoldchanges' not in df.columns:
        rename_map = {}
        for c in df.columns:
            if 'log' in c.lower() and 'fc' in c.lower():
                rename_map[c] = 'logfoldchanges'
            elif 'padj' in c.lower() or 'pval' in c.lower() or 'adj' in c.lower():
                rename_map[c] = 'pvals_adj'
        df = df.rename(columns=rename_map)
    
    if 'logfoldchanges' not in df.columns or 'pvals_adj' not in df.columns:
        print(f'  Skipping {ct}: missing required columns')
        continue
    
    df['logfoldchanges'] = pd.to_numeric(df['logfoldchanges'], errors='coerce')
    df['pvals_adj'] = pd.to_numeric(df['pvals_adj'], errors='coerce')
    
    # Significant genes
    sig = df[(df['pvals_adj'] < PVAL_THRESHOLD) & (df['logfoldchanges'].abs() > LFC_THRESHOLD)]
    if len(sig) == 0:
        continue
    
    deg_by_celltype[ct] = sig.copy()
    print(f'  {ct}: {len(sig)} sig DEGs ({(sig["logfoldchanges"] > 0).sum()} UP, {(sig["logfoldchanges"] < 0).sum()} DOWN)')

print(f'\nCell types with sig DEGs: {len(deg_by_celltype)}')

In [ ]:
# --------------------------------------------------------------------------
# 2) Count gene recurrence across cell types (UP and DOWN separately)
# --------------------------------------------------------------------------

# Build gene x celltype matrices
all_ct = list(deg_by_celltype.keys())
all_genes = set()
for ct, df in deg_by_celltype.items():
    gene_col = 'names' if 'names' in df.columns else df.columns[0]
    all_genes.update(df[gene_col].astype(str).tolist())
all_genes = sorted(all_genes)

gene_up_count = {}
gene_down_count = {}
gene_lfc = {}  # mean log2FC across cell types

for gene in all_genes:
    up_cts = []
    down_cts = []
    lfc_vals = []
    for ct, df in deg_by_celltype.items():
        gene_col = 'names' if 'names' in df.columns else df.columns[0]
        gene_df = df[df[gene_col].astype(str) == gene]
        if len(gene_df) > 0:
            lfc = gene_df['logfoldchanges'].values[0]
            lfc_vals.append(lfc)
            if lfc > 0:
                up_cts.append(ct)
            else:
                down_cts.append(ct)
    if up_cts:
        gene_up_count[gene] = (len(up_cts), up_cts)
    if down_cts:
        gene_down_count[gene] = (len(down_cts), down_cts)
    if lfc_vals:
        gene_lfc[gene] = np.mean(lfc_vals)

# Sort by frequency
up_sorted = sorted(gene_up_count.items(), key=lambda x: x[1][0], reverse=True)
down_sorted = sorted(gene_down_count.items(), key=lambda x: x[1][0], reverse=True)

print(f'Genes UP in >=1 cell type: {len(up_sorted)}')
print(f'Genes DOWN in >=1 cell type: {len(down_sorted)}')
print(f'\nTop UP genes shared across cell types:')
for g, (n, cts) in up_sorted[:20]:
    if n >= 3:
        print(f'  {g}: {n} cell types ({cts[:3]}...)')

print(f'\nTop DOWN genes shared across cell types:')
for g, (n, cts) in down_sorted[:20]:
    if n >= 3:
        print(f'  {g}: {n} cell types ({cts[:3]}...)')

In [ ]:
# --------------------------------------------------------------------------
# 3) Visualization: Frequency barplot of shared genes
# --------------------------------------------------------------------------

for direction, sorted_data, cmap_name, title_suffix in [
    ('UP', up_sorted, 'Reds', 'Up-regulated'),
    ('DOWN', down_sorted, 'Blues', 'Down-regulated'),
]:
    # Genes shared in >=2 cell types
    shared = [(g, n, cts) for g, (n, cts) in sorted_data if n >= 2]
    if len(shared) == 0:
        continue
    
    top_n = min(30, len(shared))
    top_shared = shared[:top_n]
    
    fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * top_n + 1)))
    genes = [x[0] for x in top_shared]
    counts = [x[1] for x in top_shared]
    colors = plt.cm.get_cmap(cmap_name)(np.linspace(0.4, 0.9, top_n))
    
    bars = ax.barh(range(top_n), counts, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(genes, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Cell Types', fontsize=12)
    ax.set_title(f'Top Shared {title_suffix} Genes Across Cell Types (R301 vs PBS)',
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add cell type names as annotation for top 5
    for i, (g, n, cts) in enumerate(top_shared[:5]):
        ct_str = ', '.join(cts[:5])
        if len(cts) > 5:
            ct_str += f' ...+{len(cts)-5}'
        ax.text(n + 0.1, i, ct_str, va='center', fontsize=7, color='#333333')
    
    plt.tight_layout()
    plt.savefig(os.path.join(shared_dir, f'Shared_DEG_frequency_{direction}.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    # Save table
    freq_df = pd.DataFrame([
        {'gene': g, 'n_celltypes': n, 'celltypes': ', '.join(cts), 'mean_log2FC': gene_lfc.get(g, np.nan)}
        for g, n, cts in shared
    ])
    freq_df.to_csv(os.path.join(shared_dir, f'Shared_DEG_{direction}_frequency.csv'), index=False)

In [ ]:
# --------------------------------------------------------------------------
# 4) Heatmap: log2FC of top shared genes across cell types
# --------------------------------------------------------------------------

for direction, sorted_data, cmap_name in [
    ('UP', up_sorted, 'RdBu_r'),
    ('DOWN', down_sorted, 'RdBu_r'),
]:
    shared = [(g, n, cts) for g, (n, cts) in sorted_data if n >= 3]
    if len(shared) < 3:
        continue
    
    # Top 25 shared genes
    top_genes = [x[0] for x in shared[:25]]
    
    # Build log2FC matrix: genes x cell types
    lfc_matrix = pd.DataFrame(index=top_genes, columns=all_ct, data=np.nan)
    for ct, df in deg_by_celltype.items():
        gene_col = 'names' if 'names' in df.columns else df.columns[0]
        for _, row in df.iterrows():
            g = str(row[gene_col])
            if g in top_genes:
                lfc_matrix.loc[g, ct] = row['logfoldchanges']
    
    # Drop rows and columns that are all NaN
    lfc_matrix = lfc_matrix.dropna(how='all', axis=0).dropna(how='all', axis=1)
    if lfc_matrix.empty:
        continue
    lfc_matrix = lfc_matrix.fillna(0)
    
    # Sort rows by sum of |LFC|, columns alphabetically
    lfc_matrix['abs_sum'] = lfc_matrix.abs().sum(axis=1)
    lfc_matrix = lfc_matrix.sort_values('abs_sum', ascending=False).drop(columns='abs_sum')
    
    fig, ax = plt.subplots(
        figsize=(max(10, 0.3 * lfc_matrix.shape[1] + 3),
                 max(5, 0.35 * lfc_matrix.shape[0] + 2))
    )
    sns.heatmap(lfc_matrix, cmap=cmap_name, center=0, linewidths=0.3, linecolor='white',
                cbar_kws={'label': 'log2 Fold Change (R301 vs PBS)'}, ax=ax)
    ax.set_title(f'{direction}-regulated DEGs Shared Across Cell Types',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(shared_dir, f'Shared_DEG_heatmap_{direction}.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# --------------------------------------------------------------------------
# 5) GO enrichment of cross-celltype shared genes
# --------------------------------------------------------------------------

for direction, sorted_data, title_suffix in [
    ('UP', up_sorted, 'Up-regulated'),
    ('DOWN', down_sorted, 'Down-regulated'),
]:
    shared = [(g, n, cts) for g, (n, cts) in sorted_data if n >= 3]
    if len(shared) < 5:
        print(f'Too few shared {direction} genes for enrichment (need >=5, got {len(shared)})')
        continue
    
    shared_genes = [x[0] for x in shared]
    print(f'\nRunning GO enrichment for {len(shared_genes)} shared {direction} genes...')
    
    try:
        enr = gp.enrichr(
            gene_list=shared_genes,
            gene_sets=['GO_Biological_Process_2023'],
            organism='mouse',
            outdir=None,
        )
        if enr is not None and enr.results is not None and not enr.results.empty:
            res = enr.results.copy()
            res.to_csv(os.path.join(shared_dir, f'Shared_DEG_GO_enrichment_{direction}.csv'), index=False)
            
            # Dotplot of top 15 terms
            top = res.head(15).copy()
            top['Adjusted P-value'] = top['Adjusted P-value'].astype(float)
            top['-log10(q)'] = -np.log10(top['Adjusted P-value'] + 1e-300)
            top = top.sort_values('-log10(q)')
            
            fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(top) + 1.5)))
            sca = ax.scatter(
                top['-log10(q)'], top['Term'],
                s=60 + 10 * top['Overlap'].str.split('/', expand=True)[0].astype(float),
                c=top['Combined Score'].astype(float) if 'Combined Score' in top.columns else top['-log10(q)'],
                cmap='viridis', alpha=0.9, edgecolor='none'
            )
            ax.set_xlabel('-log10(Adjusted P-value)', fontsize=12)
            ax.set_title(f'GO BP: Shared {title_suffix} Genes Across Cell Types',
                         fontsize=13, fontweight='bold')
            cbar = plt.colorbar(sca, ax=ax, pad=0.02)
            cbar.set_label('Combined Score')
            sns.despine(ax=ax, left=False, bottom=False)
            plt.tight_layout()
            plt.savefig(os.path.join(shared_dir, f'Shared_DEG_GO_enrichment_{direction}.pdf'),
                        dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
            print(f'  Top 5 GO terms:')
            for _, r in top.head(5).iterrows():
                print(f'    {r["Term"]}: q={r["Adjusted P-value"]:.2e}')
        else:
            print(f'  No enrichment results for {direction} genes')
    except Exception as e:
        print(f'  Enrichment failed for {direction}: {e}')

print(f'\nShared DEG analysis saved to: {shared_dir}')

### 7.3 Gene Signature Scoring at Sample Level

定义治疗相关的基因签名（免疫激活、T细胞耗竭、IFN响应、血管生成、ECM重塑、抗原呈递、炎症），
在细胞水平评分后聚合到样本水平，比较 PBS vs R301。

In [ ]:
# ==============================================================================
# 7.3 Gene Signature Scoring at Sample Level
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sig_dir = os.path.join(RESULTS_DIR, 'sample_signatures')
os.makedirs(sig_dir, exist_ok=True)

# --------------------------------------------------------------------------
# Define treatment-relevant gene signatures
# --------------------------------------------------------------------------
treatment_signatures = {
    'Immune_Activation': [
        'Ifng', 'Gzmb', 'Prf1', 'Tbx21', 'Eomes', 'Cd69', 'Icos',
        'Il2ra', 'Tnf', 'Ccl3', 'Ccl4', 'Xcl1'
    ],
    'T_Cell_Exhaustion': [
        'Pdcd1', 'Lag3', 'Havcr2', 'Tigit', 'Ctla4', 'Tox', 'Entpd1', 'Eomes'
    ],
    'IFN_Response': [
        'Ifit1', 'Ifit3', 'Isg15', 'Stat1', 'Stat2', 'Irf7', 'Irf9',
        'Oasl2', 'Rsad2', 'Ifih1', 'Mx1', 'Usp18'
    ],
    'Angiogenesis': [
        'Vegfa', 'Vegfb', 'Kdr', 'Flt1', 'Angpt1', 'Angpt2', 'Tek',
        'Pdgfa', 'Pdgfb', 'Pdgfra', 'Pdgfrb', 'Nrp1'
    ],
    'ECM_Remodeling': [
        'Col1a1', 'Col3a1', 'Col5a2', 'Fbn1', 'Mmp2', 'Mmp9', 'Mmp14',
        'Lox', 'Loxl2', 'Fn1', 'Timp1', 'Postn'
    ],
    'Antigen_Presentation': [
        'B2m', 'H2-K1', 'H2-D1', 'H2-Aa', 'H2-Ab1', 'H2-Eb1',
        'Cd74', 'Tap1', 'Tap2', 'Tapbp', 'Ciita', 'Psmb8', 'Psmb9'
    ],
    'Inflammatory_Response': [
        'Il1b', 'Il6', 'Tnf', 'Cxcl1', 'Cxcl2', 'Cxcl3',
        'Ccl2', 'Ccl7', 'Ptgs2', 'Nfkbia', 'Socs3', 'Cxcl10'
    ],
}

# Score each cell
for sig_name, genes in treatment_signatures.items():
    genes_avail = [g for g in genes if g in adata.raw.var_names]
    if len(genes_avail) >= 3:
        sc.tl.score_genes(
            adata, gene_list=genes_avail, score_name=sig_name,
            use_raw=True, ctrl_size=min(50, len(genes_avail) * 2), random_state=42
        )
        print(f'Scored {sig_name}: {len(genes_avail)}/{len(genes)} genes')

sig_score_cols = [s for s in treatment_signatures.keys() if s in adata.obs.columns]
print(f'\n{len(sig_score_cols)} signature scores computed')

In [ ]:
# --------------------------------------------------------------------------
# Aggregate to sample level & visualize
# --------------------------------------------------------------------------

# Per-sample mean of signature scores
samp_sig = adata.obs.groupby('sampleID')[sig_score_cols].mean()
samp_sig = samp_sig.join(sample_to_group)
samp_sig = samp_sig.sort_values('group')

# ---- A) Heatmap: Z-scored signatures across samples ----
z_sig = (samp_sig[sig_score_cols] - samp_sig[sig_score_cols].mean()) / samp_sig[sig_score_cols].std()
z_sig = z_sig.fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(z_sig.T.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(z_sig)))
ax.set_xticklabels([f'{r.name} ({r["group"]})' for _, r in samp_sig.iterrows()],
                   rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(sig_score_cols)))
ax.set_yticklabels([s.replace('_', ' ') for s in sig_score_cols], fontsize=10)
ax.set_title('Treatment-Associated Gene Signatures by Sample', fontsize=13, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Z-score')
plt.tight_layout()
plt.savefig(os.path.join(sig_dir, 'Sample_Signatures_heatmap.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ---- B) Per-signature boxplots with MWU + FDR ----
n_cols = 4
n_rows = int(np.ceil(len(sig_score_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

sig_mw_results = []
for i, sc_name in enumerate(sig_score_cols):
    ax = axes[i]
    sns.boxplot(data=samp_sig, x='group', y=sc_name, order=existing_groups,
                palette=color, width=0.5, ax=ax)
    sns.stripplot(data=samp_sig, x='group', y=sc_name, order=existing_groups,
                   color='black', size=7, jitter=0.12, alpha=0.85, ax=ax)
    
    a_vals = samp_sig.loc[samp_sig['group'] == 'R301', sc_name].values
    b_vals = samp_sig.loc[samp_sig['group'] == 'PBS', sc_name].values
    if len(a_vals) >= 2 and len(b_vals) >= 2:
        u, p = mannwhitneyu(a_vals, b_vals, alternative='two-sided')
        sig_mw_results.append({'signature': sc_name, 'u_stat': u, 'p_value': p})
    else:
        sig_mw_results.append({'signature': sc_name, 'u_stat': np.nan, 'p_value': np.nan})
    
    ax.set_title(sc_name.replace('_', ' '), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('Mean Score per Sample')

for j in range(len(sig_score_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(sig_dir, 'Sample_Signatures_boxplot.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# FDR correction
sig_mw_df = pd.DataFrame(sig_mw_results)
sig_mw_df['p_fdr'] = multipletests(sig_mw_df['p_value'].fillna(1.0).values, method='fdr_bh')[1]
sig_mw_df.to_csv(os.path.join(sig_dir, 'Sample_Signatures_MWU.csv'), index=False)
print('\nSample-Level Signature Comparison (R301 vs PBS):')
print(sig_mw_df.round(4).to_string(index=False))

print(f'\nSample signature figures saved to: {sig_dir}')

### 7.4 Functional Module Scoring Across All Compartments

在 T/NK、髓系、基质等各 compartments 中分别定义功能模块，
计算模块评分并在对应细胞群内比较 PBS vs R301，汇总为全局热图。

In [ ]:
# ==============================================================================
# 7.4 Functional Module Scoring Across All Compartments
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

module_dir = os.path.join(RESULTS_DIR, 'compartment_modules')
os.makedirs(module_dir, exist_ok=True)

# --------------------------------------------------------------------------
# 1) Classify cell types into compartments
# --------------------------------------------------------------------------
compartment_map = {
    # T / NK
    'Cytotoxic NK/NKT-like cells': 'T_NK',
    'Cytotoxic CD8 T cells': 'T_NK',
    'Activated T cells': 'T_NK',
    # Myeloid
    'Neutrophils': 'Myeloid',
    'IL1B+ inflammatory macrophages': 'Myeloid',
    'CCL2+ inflammatory macrophages': 'Myeloid',
    'C1q+ antigen-presenting macrophages': 'Myeloid',
    'MRC1+ resident-like macrophages': 'Myeloid',
    'CTSK+ osteoclast-like macrophages': 'Myeloid',
    # DC
    'pDCs': 'Myeloid',
    'CLEC9A+ cDCs': 'Myeloid',
    'CCR7+ mature DCs': 'Myeloid',
    # Stromal
    'ECM-remodeling fibroblasts': 'Stromal',
    'Vascular endothelial cells': 'Stromal',
    # Mast / B
    'Mast cells': 'Myeloid',
    'B cells': 'T_NK',  # small population, group with lymphoid
}

# Malignant cells keep their own fine-grained types
malignant_celltypes = [ct for ct in adata.obs['celltype'].unique() if 'malignant' in ct.lower()
                       or ct.startswith('Prolif') or ct.startswith('Cycling')
                       or ct.startswith('IFN') or ct.startswith('Hypoxic')
                       or ct.startswith('Growth') or ct.startswith('Translation')
                       or ct.startswith('CDH13') or ct.startswith('Mesenchymal')]

adata.obs['compartment'] = adata.obs['celltype'].map(compartment_map)
# Unmapped cell types keep their original name (e.g., malignant subtypes)
adata.obs['compartment'] = adata.obs['compartment'].fillna('Malignant')

print('Compartment distribution:')
print(adata.obs['compartment'].value_counts())

In [ ]:
# --------------------------------------------------------------------------
# 2) Define compartment-specific functional modules
# --------------------------------------------------------------------------

compartment_modules = {
    'T_NK': {
        'Cytotoxicity': ['Gzma', 'Gzmb', 'Gzmk', 'Prf1', 'Ctsw', 'Nkg7', 'Fasl', 'Tnf'],
        'Exhaustion': ['Pdcd1', 'Lag3', 'Havcr2', 'Tigit', 'Ctla4', 'Tox', 'Entpd1'],
        'Activation': ['Cd69', 'Icos', 'Cd44', 'Il2ra', 'Ifng', 'Cd28', 'Cd40lg'],
        'IFN_Response': ['Ifit1', 'Ifit3', 'Isg15', 'Stat1', 'Irf7', 'Irf9', 'Oasl2', 'Rsad2'],
        'Tissue_Residency': ['Itgae', 'Cd69', 'Cxcr6', 'Hobit', 'Runx3', 'Prdm1'],
    },
    'Myeloid': {
        'M1_Polarization': ['Il1b', 'Tnf', 'Nos2', 'Cxcl9', 'Cxcl10', 'Ccl3', 'Cd80', 'Cd86', 'Ciita'],
        'M2_Polarization': ['Mrc1', 'Arg1', 'Chil3', 'Cd163', 'Il10', 'Tgfb1', 'Ccl22', 'Stab1', 'F13a1'],
        'Phagocytosis': ['Fcgr1', 'Fcgr3', 'Mertk', 'Cd68', 'Trem2', 'Itgam', 'Marco', 'Msr1', 'Cd36'],
        'Antigen_Presentation': ['H2-Aa', 'H2-Ab1', 'H2-Eb1', 'Cd74', 'B2m', 'H2-K1', 'H2-D1', 'Tap1', 'Ciita'],
        'Chemokine_Production': ['Ccl2', 'Ccl7', 'Ccl8', 'Ccl12', 'Cxcl1', 'Cxcl2', 'Cxcl3', 'Cxcl9', 'Cxcl10'],
        'Inflammasome': ['Nlrp3', 'Il1b', 'Il18', 'Casp1', 'Pycard', 'Aim2'],
    },
    'Stromal': {
        'ECM_Production': ['Col1a1', 'Col1a2', 'Col3a1', 'Col5a2', 'Fbn1', 'Fn1', 'Postn', 'Tnc'],
        'ECM_Degradation': ['Mmp2', 'Mmp9', 'Mmp14', 'Timp1', 'Timp2', 'Lox', 'Loxl2'],
        'Angiogenesis': ['Pecam1', 'Cdh5', 'Eng', 'Vwf', 'Flt1', 'Kdr', 'Angpt2', 'Nrp1', 'Egfl7'],
        'Inflammatory_CAF': ['Il6', 'Ccl2', 'Cxcl1', 'Cxcl12', 'Ptgs2', 'Lif', 'Csf1'],
    },
}

# For 'Malignant', reuse existing module scores or compute additional ones
# This complements the earlier malignant module scoring (Section 6) with
# treatment-focused modules scored only in malignant cells
compartment_modules['Malignant'] = {
    'EMT_Score': ['Vim', 'Fn1', 'Col1a1', 'Col1a2', 'Prrx1', 'Zeb1', 'Zeb2', 'Snai1', 'Snai2', 'Twist1', 'Cdh13', 'Slit2'],
    'Immune_Evasion': ['Cd274', 'Pdcd1lg2', 'Cd47', 'Serpinb9', 'Fasl', 'Ccl2', 'Cxcl1', 'Vegfa'],
    'Stress_Response': ['Ddit3', 'Hspa5', 'Atf4', 'Ern1', 'Xbp1', 'Hsp90b1', 'Hspa1a', 'Hspa1b', 'Dnajb1'],
}

In [ ]:
# --------------------------------------------------------------------------
# 3) Score modules within each compartment
# --------------------------------------------------------------------------

all_module_results = {}  # {compartment: {module_name: {'PBS_mean': ..., 'R301_mean': ..., 'p_value': ...}}}

for comp, modules in compartment_modules.items():
    # Subset to cells in this compartment
    comp_adata = adata[adata.obs['compartment'] == comp].copy()
    if comp_adata.n_obs < 50:
        print(f'Skipping {comp}: only {comp_adata.n_obs} cells')
        continue
    
    comp_results = {}
    scored_this_comp = []

    for mod_name, genes in modules.items():
        genes_avail = [g for g in genes if g in adata.raw.var_names]
        if len(genes_avail) < 3:
            continue

        mod_score_col = f'{comp}_{mod_name}'
        sc.tl.score_genes(
            comp_adata, gene_list=genes_avail, score_name=mod_score_col,
            use_raw=True, ctrl_size=min(50, len(genes_avail) * 2), random_state=42
        )
        
        # Aggregate to sample level for MWU
        samp_mean = comp_adata.obs.groupby('sampleID')[mod_score_col].mean()
        samp_mean = samp_mean.to_frame().join(sample_to_group)
        
        pbs_vals = samp_mean.loc[samp_mean['group'] == 'PBS', mod_score_col].values
        r301_vals = samp_mean.loc[samp_mean['group'] == 'R301', mod_score_col].values
        
        if len(pbs_vals) >= 2 and len(r301_vals) >= 2:
            u, p = mannwhitneyu(r301_vals, pbs_vals, alternative='two-sided')
        else:
            u, p = np.nan, np.nan
        
        comp_results[mod_name] = {
            'PBS_mean': float(pbs_vals.mean()) if len(pbs_vals) > 0 else np.nan,
            'R301_mean': float(r301_vals.mean()) if len(r301_vals) > 0 else np.nan,
            'delta': float(r301_vals.mean() - pbs_vals.mean()) if (len(r301_vals) > 0 and len(pbs_vals) > 0) else np.nan,
            'p_value': p,
            'n_genes_used': len(genes_avail),
        }
        scored_this_comp.append(mod_name)
    
    all_module_results[comp] = comp_results
    print(f'{comp}: scored {len(scored_this_comp)} modules' + 
          f' ({comp_adata.n_obs} cells)')

print(f'\nTotal compartments scored: {len(all_module_results)}')

In [ ]:
# --------------------------------------------------------------------------
# 4) Comprehensive heatmap of all compartment modules
# --------------------------------------------------------------------------

# Flatten into a dataframe
heatmap_rows = []
for comp, modules in all_module_results.items():
    for mod_name, stats in modules.items():
        heatmap_rows.append({
            'Module': f'{comp}: {mod_name}',
            'Compartment': comp,
            'delta': stats['delta'],
            'p_value': stats['p_value'],
            'PBS_mean': stats['PBS_mean'],
            'R301_mean': stats['R301_mean'],
        })

heat_df = pd.DataFrame(heatmap_rows).set_index('Module')
if not heat_df.empty:
    # FDR correction across all module tests
    from statsmodels.stats.multitest import multipletests
    pvals = heat_df['p_value'].fillna(1.0).values
    _, qvals, _, _ = multipletests(pvals, method='fdr_bh')
    heat_df['q_fdr'] = qvals
else:
    heat_df['q_fdr'] = np.nan

heat_df.to_csv(os.path.join(module_dir, 'Compartment_Modules_Summary.csv'))

# Plot heatmap of delta (R301 - PBS)
plot_data = heat_df[['delta']].copy()
plot_data['Module'] = plot_data.index

fig, ax = plt.subplots(figsize=(6, max(5, 0.4 * len(plot_data) + 1)))

colors = ['#d62728' if d > 0 else '#1f77b4' for d in plot_data['delta']]
bars = ax.barh(range(len(plot_data)), plot_data['delta'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['Module'], fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Delta Score (R301 - PBS)', fontsize=12)
ax.set_title('Compartment-Level Functional Module Changes After Treatment',
             fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add significance stars
for i, (_, row) in enumerate(heat_df.iterrows()):
    if row['q_fdr'] < 0.001:
        star = '***'
    elif row['q_fdr'] < 0.01:
        star = '**'
    elif row['q_fdr'] < 0.05:
        star = '*'
    else:
        continue
    x_pos = row['delta'] + (0.02 if row['delta'] >= 0 else -0.02)
    ha = 'left' if row['delta'] >= 0 else 'right'
    ax.text(x_pos, i, star, va='center', ha=ha, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(module_dir, 'Compartment_Modules_delta.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Print significant modules
print('\nSignificant functional module changes (q < 0.05):')
sig_modules = heat_df[heat_df['q_fdr'] < 0.05]
if len(sig_modules) > 0:
    for idx, row in sig_modules.iterrows():
        direction = 'UP' if row['delta'] > 0 else 'DOWN'
        print(f'  {idx}: delta={row["delta"]:.3f}, q={row["q_fdr"]:.3f} [{direction}]')
else:
    print('  No modules reach significance at q < 0.05 (may be due to n=3 per group)')

print(f'\nCompartment module results saved to: {module_dir}')

In [ ]:
# --------------------------------------------------------------------------
# 5) Per-compartment summary heatmaps (more detailed view)
# --------------------------------------------------------------------------

for comp, modules in all_module_results.items():
    if len(modules) < 2:
        continue
    
    fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(modules) + 1)))
    
    mod_names = list(modules.keys())
    deltas = [modules[m]['delta'] for m in mod_names]
    pvals = [modules[m]['p_value'] for m in mod_names]
    pbs_vals = [modules[m]['PBS_mean'] for m in mod_names]
    r301_vals = [modules[m]['R301_mean'] for m in mod_names]
    
    # Delta heatmap-like barhor chart
    vmax = max(abs(min(deltas or [0])), abs(max(deltas or [0])), 0.01) * 1.2
    colors_list = plt.cm.RdBu_r((np.array(deltas) / vmax + 1) / 2)
    
    ax.barh(range(len(mod_names)), deltas, color=colors_list, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(mod_names)))
    ax.set_yticklabels(mod_names, fontsize=10)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Delta Score (R301 - PBS)')
    ax.set_title(f'{comp} Compartment: Functional Module Changes',
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Annotate with p-values
    for i, (m, d, p) in enumerate(zip(mod_names, deltas, pvals)):
        x_pos = d + (vmax * 0.03 if d >= 0 else -vmax * 0.03)
        ha = 'left' if d >= 0 else 'right'
        p_str = f'p={p:.3f}' if not np.isnan(p) else 'n.s.'
        ax.text(x_pos, i, p_str, va='center', ha=ha, fontsize=8, color='#333333')
    
    plt.tight_layout()
    plt.savefig(os.path.join(module_dir, f'Module_delta_{comp}.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Per-compartment module figures saved to: {module_dir}')

In [ ]:
# ==============================================================================
# Save updated AnnData with all scores
# ==============================================================================
adata.write(DATA_DIR + '/Step3-sce_annotated.h5ad', compression='gzip')
print('Updated adata with all module/signature scores saved.')

---

### 7.5 Enrichr Cross-Celltype Summary — Global Transcriptional Reprogramming Patterns

汇总所有 cell type 的 Enrichr GO 富集结果，识别 R301 治疗在多种细胞类型中
共同激活或抑制的生物学通路。

In [ ]:
# ==============================================================================
# 7.5 Enrichr Cross-Celltype Summary
# ==============================================================================

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

enrichr_summary_dir = os.path.join(RESULTS_DIR, 'enrichr_summary')
os.makedirs(enrichr_summary_dir, exist_ok=True)

DEG_DIR = os.path.join(BASE_DIR, 'DEG_Enrichr_all_comparisons/R301_vs_PBS')

def parse_ct(fname):
    return os.path.basename(fname).replace('_Enrichr_up.csv','').replace('_Enrichr_down.csv','').replace('_',' ').replace('  ',' ')

# ---- Load all Enrichr results ----
all_data = []
for direction in ['up', 'down']:
    files = sorted(glob.glob(os.path.join(DEG_DIR, f'*_Enrichr_{direction}.csv')))
    for f in files:
        ct = parse_ct(f)
        df = pd.read_csv(f)
        df['celltype'] = ct
        df['direction'] = direction
        all_data.append(df)

enr_all = pd.concat(all_data, ignore_index=True)
enr_all['Adjusted P-value'] = pd.to_numeric(enr_all['Adjusted P-value'], errors='coerce')
enr_all['-log10q'] = -np.log10(enr_all['Adjusted P-value'] + 1e-300)
enr_sig = enr_all[enr_all['Adjusted P-value'] < 0.05].copy()
print(f'Cell types with Enrichr data: {enr_all["celltype"].nunique()}')
print(f'Total significant enriched terms (q<0.05): {len(enr_sig)}')
print(f'  UP: {(enr_sig["direction"]=="up").sum()}, DOWN: {(enr_sig["direction"]=="down").sum()}')

# ---- Term frequency across cell types ----
term_freq_up = enr_sig[enr_sig['direction']=='up'].groupby('Term')['celltype'].nunique().sort_values(ascending=False)
term_freq_down = enr_sig[enr_sig['direction']=='down'].groupby('Term')['celltype'].nunique().sort_values(ascending=False)

print('\nTop UP terms by cell-type recurrence:')
for t in term_freq_up.head(10).index:
    cts = enr_sig[(enr_sig['Term']==t) & (enr_sig['direction']=='up')]['celltype'].unique()
    print(f'  [{len(cts)} CTs] {t[:80]}')

print('\nTop DOWN terms by cell-type recurrence:')
for t in term_freq_down.head(10).index:
    cts = enr_sig[(enr_sig['Term']==t) & (enr_sig['direction']=='down')]['celltype'].unique()
    print(f'  [{len(cts)} CTs] {t[:80]}')

In [ ]:
# ---- Visualization A: Most recurring terms (UP vs DOWN side-by-side) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (freq, direction, title, cmap_name) in zip(
    axes,
    [(term_freq_up, 'up', 'UP in R301 vs PBS', 'Reds'),
     (term_freq_down, 'down', 'DOWN in R301 vs PBS', 'Blues')]
):
    top = freq.head(20)
    colors = plt.get_cmap(cmap_name)(np.linspace(0.5, 0.95, len(top)))
    
    ax.barh(range(len(top)), top.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([t[:80] for t in top.index], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Cell Types with Significant Enrichment', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(top.values):
        ax.text(v + 0.1, i, str(v), va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(enrichr_summary_dir, 'Enrichr_Recurring_Terms_UP_DOWN.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: Enrichr_Recurring_Terms_UP_DOWN.pdf')

In [ ]:
# ---- Visualization B: Heatmap of shared terms × cell types ----
for direction, freq_data, cmap_name, direction_label in [
    ('up', term_freq_up, 'Reds', 'UP-regulated'),
    ('down', term_freq_down, 'Blues', 'DOWN-regulated'),
]:
    top_terms = freq_data.head(15).index.tolist()
    heat_data = enr_sig[(enr_sig['Term'].isin(top_terms)) & (enr_sig['direction']==direction)]
    mat = heat_data.pivot_table(
        index='Term', columns='celltype', values='-log10q', aggfunc='max', fill_value=0
    )
    mat = mat.loc[mat.sum(axis=1) > 0, mat.sum(axis=0) > 0]
    if mat.empty:
        continue
    mat.index = [t[:80] for t in mat.index]
    
    fig, ax = plt.subplots(figsize=(max(12, 0.35 * mat.shape[1] + 3), max(6, 0.4 * mat.shape[0] + 2)))
    sns.heatmap(mat, cmap=cmap_name, annot=True, fmt='.1f', linewidths=0.3, linecolor='white',
                cbar_kws={'label': '-log10(Adjusted P-value)'}, ax=ax, annot_kws={'fontsize': 8})
    ax.set_title(f'Shared GO Terms: {direction_label} in R301 vs PBS\n(Top terms by cell-type recurrence)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(enrichr_summary_dir, f'Enrichr_SharedTerms_Heatmap_{direction.upper()}.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved: Enrichr_SharedTerms_Heatmap_{direction.upper()}.pdf')

In [ ]:
# ---- Visualization C: Top 3 enriched terms per cell type (dotplot) ----
for direction, title_suffix in [('up', 'UP-regulated'), ('down', 'DOWN-regulated')]:
    top_per_ct = enr_sig[enr_sig['direction'] == direction].copy()
    top_per_ct = top_per_ct.sort_values('Adjusted P-value').groupby('celltype').head(3)
    if top_per_ct.empty:
        continue
    
    top_per_ct['Term_short'] = top_per_ct['Term'].str[:70]
    # Extract hit count from Overlap (e.g., "12/200")
    try:
        top_per_ct['n_genes'] = top_per_ct['Overlap'].astype(str).str.split('/').str[0].astype(float)
    except:
        top_per_ct['n_genes'] = 1.0
    
    unique_terms = top_per_ct.drop_duplicates('Term').sort_values('Adjusted P-value')['Term_short'].tolist()
    celltypes_sorted = sorted(top_per_ct['celltype'].unique())
    ct_to_x = {ct: i for i, ct in enumerate(celltypes_sorted)}
    term_to_y = {t: i for i, t in enumerate(unique_terms)}
    
    fig, ax = plt.subplots(figsize=(12, max(5, 0.35 * len(unique_terms) + 2)))
    scatter = ax.scatter(
        [ct_to_x[ct] for ct in top_per_ct['celltype']],
        [term_to_y[t] for t in top_per_ct['Term_short']],
        s=30 + 12 * top_per_ct['n_genes'].values,
        c=top_per_ct['-log10q'].values,
        cmap='Reds' if direction == 'up' else 'Blues',
        alpha=0.85, edgecolor='grey', linewidth=0.3
    )
    ax.set_xticks(range(len(celltypes_sorted)))
    ax.set_xticklabels(celltypes_sorted, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(unique_terms)))
    ax.set_yticklabels(unique_terms, fontsize=7)
    ax.set_xlim(-0.5, len(celltypes_sorted) - 0.5)
    ax.set_ylim(-0.5, len(unique_terms) - 0.5)
    ax.invert_yaxis()
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label('-log10(Adjusted P-value)')
    ax.set_title(f'Top 3 {title_suffix} GO Terms per Cell Type (R301 vs PBS)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(enrichr_summary_dir, f'Enrichr_Top3_Dotplot_{direction.upper()}.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

print(f'\nEnrichr summary saved to: {enrichr_summary_dir}')

In [ ]:
# ---- Save comprehensive cross-celltype summary table ----
summary_rows = []
for direction, freq_data in [('up', term_freq_up), ('down', term_freq_down)]:
    for term, n_cts in freq_data.items():
        cts = enr_sig[(enr_sig['Term']==term) & (enr_sig['direction']==direction)]['celltype'].unique()
        mean_q = enr_sig[(enr_sig['Term']==term) & (enr_sig['direction']==direction)]['Adjusted P-value'].mean()
        summary_rows.append({
            'direction': direction,
            'term': term,
            'n_celltypes': n_cts,
            'celltypes': ', '.join(cts),
            'mean_q': mean_q,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(['direction', 'n_celltypes', 'mean_q'],
                                      ascending=[True, False, True])
summary_df.to_csv(os.path.join(enrichr_summary_dir, 'Enrichr_CrossCelltype_Summary.csv'), index=False)

print('\n=== Enrichr Cross-Celltype Summary ===')
print(f'UP: Top terms = {term_freq_up.head(5).index.tolist()}')
print(f'DOWN: Top terms = {term_freq_down.head(5).index.tolist()}')
print(f'\nKey conclusion:')
print(f'  R301 broadly UP-regulates IFN/antiviral programs across {term_freq_up.iloc[0]} cell types')
print(f'  R301 broadly DOWN-regulates inflammatory signaling across {term_freq_down.iloc[0]} cell types')

---

## Target Analysis

Analysis of VEGF and PD1 pathway gene expression across cell populations and treatment groups.

### Setup and Population Definition

---

> ⚠️ **以下 Target Analysis / Module Analysis cells (cells 133-135) 包含其他项目的硬编码 group 名**
> (NC, AK112, CD47, RC148)，在当前 PBS vs R301 数据上无法直接运行。
> 如需在当前项目中使用，需替换为 PBS/R301 的 group 名并调整分析逻辑。
> 可参考 Section 7.3 (Sample-Level Gene Signature Scoring) 和 Section 7.4 (Compartment Modules)
> 中已有的治疗响应分析方法。

---


In [ ]:
adata = sc.read(DATA_DIR+ '/Step3-sce_annotated.h5ad')
adata

In [ ]:
# ==============================================================================
# Upgraded VEGF/PD1 Target Analysis Module
# ==============================================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# 导入 P 值自动标注工具
try:
    from statannotations.Annotator import Annotator
except ImportError:
    print("⚠️ 未找到 statannotations。请在终端运行: pip install statannotations")
    Annotator = None
# ==============================================================================
# 1. 目录创建与靶点基因定义
# ==============================================================================
target_dir = os.path.join(RESULTS_DIR, 'Target_analysis')
os.makedirs(target_dir, exist_ok=True)

# 定义双抗药物机制相关的核心基因集
target_genes_dict = {
      'VEGF_Pathway': ['Vegfa', 'Vegfb', 'Kdr', 'Flt1', 'Pgf'],
      'PD1_Pathway': ['Cd274', 'Pdcd1', 'Pdcd1lg2'],
      'CD47_Pathway': ['Cd47', 'Sirpa', 'Sirc1'],  # 新增
      'Immune_Checkpoints': ['Ctla4', 'Lag3', 'Havcr2', 'Tigit', 'Cd244'],
      'Angiogenesis': ['Angpt1', 'Angpt2', 'Tek', 'Pdgfa', 'Pdgfb'],
  }

# 展平所有基因并检查在 adata 中的存在情况
all_target_genes = list(set([g for genes in target_genes_dict.values() for g in genes]))
available_genes = [g for g in all_target_genes if g in adata.raw.var_names]
missing_genes = [g for g in all_target_genes if g not in adata.raw.var_names]

print('='*70)
print('🎯 Upgraded VEGF/PD1 Target Analysis')
print('='*70)
print(f'Output directory: {target_dir}')
print(f'Target genes defined: {len(all_target_genes)}')
print(f'Available in data: {len(available_genes)}')
if missing_genes:
    print(f'Missing genes: {missing_genes}')

# ==============================================================================
# 2. 基于样本 (Sample-level) 的精准靶点表达统计
# ==============================================================================
print("\n[Target Analysis] Calculating Sample-level statistics...")

# 精准定义：只在具有生物学意义的细胞群中看特定靶点
specific_targets = {
      'Vegfa': ['Malignant cells', 'Fibroblasts', 'Mrc1+ TAMs', 'Endothelial cells'],
      'Cd274': ['Malignant cells', 'Mrc1+ TAMs', 'cDCs', 'Monocytes'],
      'Pdcd1': ['T cells', 'NK cells'],
      'Havcr2': ['T cells', 'NK cells', 'cDCs'],
      'Kdr': ['Endothelial cells'],
      'Cd47': ['Malignant cells', 'Monocytes', 'Mrc1+ TAMs', 'Ccl8+ TAMs'],  # 配体
      'Sirpa': ['Mrc1+ TAMs', 'Ccl8+ TAMs', 'Monocytes'],  # 受体（巨噬细胞）
  }

sample_stats = []

for gene, target_celltypes in specific_targets.items():
    if gene not in adata.raw.var_names:
        continue
        
    expr = adata.raw[:, gene].X.toarray().flatten()
    
    for celltype in target_celltypes:
        if celltype not in adata.obs['celltype'].values:
            continue
            
        # 按样本(sampleID)级别汇总，用于提供统计学检验所需的独立重复
        for sample in adata.obs['sampleID'].unique():
            # 获取该样本的组别
            sample_df = adata.obs[adata.obs['sampleID'] == sample]
            if sample_df.empty:
                continue
            group = sample_df['group'].iloc[0]
            
            mask = (adata.obs['celltype'] == celltype) & (adata.obs['sampleID'] == sample)
            cell_count = mask.sum()
            
            # 过滤掉细胞数量太少的样本点（避免极小样本造成的百分比伪影）
            if cell_count >= 5: 
                group_expr = expr[mask]
                pct_pos = (group_expr > 0).mean() * 100
                mean_exp = group_expr.mean()
                
                sample_stats.append({
                    'Gene': gene,
                    'CellType': celltype,
                    'Sample': sample,
                    'Group': group,
                    'Cell_Count': cell_count,
                    'Pct_Positive': pct_pos,
                    'Mean_Expr': mean_exp
                })

df_sample_stats = pd.DataFrame(sample_stats)
df_sample_stats.to_csv(os.path.join(target_dir, 'Target_SampleLevel_Stats.csv'), index=False)

# ==============================================================================
# 3. 绘制带有误差棒与 P 值的靶点分布图 (Boxplot + Stripplot)
# ==============================================================================
print("\n[Target Analysis] Generating statistical plots with p-values...")

stat_pairs = [
      ("NC", "AK112"),         # 主效应
      ("NC", "CD47"),          # CD47单药效应
      ("NC", "AK112+CD47"),    # 联合治疗效应
      ("AK112", "AK112+CD47"), # 联合 vs AK112单药
      ("CD47", "AK112+CD47"),  # 联合 vs CD47单药
  ]

for gene in specific_targets.keys():
    if gene not in df_sample_stats['Gene'].values:
        continue
        
    df_plot = df_sample_stats[df_sample_stats['Gene'] == gene]
    if df_plot.empty:
        continue
        
    g = sns.catplot(
        data=df_plot, x='Group', y='Pct_Positive', col='CellType', 
        kind='box', order=treatment_order, palette=group_colors,
        height=4, aspect=0.8, sharey=False
    )
    
    # 添加底层散点，展示真实样本点
    g.map_dataframe(sns.stripplot, x='Group', y='Pct_Positive', 
                    order=treatment_order, color='black', alpha=0.6, jitter=True)
    
    g.set_axis_labels("", f"% {gene}+ Cells")
    g.set_titles("{col_name}")
    g.fig.suptitle(f'{gene} Expression by Sample', y=1.05, fontweight='bold')
    
    # 自动添加显著性检验星号
    if Annotator is not None:
        for ax, celltype in zip(g.axes.flat, df_plot['CellType'].unique()):
            data_subset = df_plot[df_plot['CellType'] == celltype]
            # 确保每组都有数据，防止报错
            if len(data_subset['Group'].unique()) > 1:
                try:
                    annotator = Annotator(ax, stat_pairs, data=data_subset, 
                                          x='Group', y='Pct_Positive', order=treatment_order)
                    annotator.configure(test='t-test_ind', text_format='star', loc='inside')
                    annotator.apply_and_annotate()
                except Exception as e:
                    print(f"  Note: Skipped annotation for {gene} in {celltype} (too few samples)")

    plt.savefig(os.path.join(target_dir, f'Target_{gene}_SampleStats.pdf'), bbox_inches='tight', dpi=300)
    plt.close()

# ==============================================================================
# 4. 嵌套 DotPlot：展示 Celltype x Group 的靶点全景动态
# ==============================================================================
print("\n[Target Analysis] Generating Nested DotPlot...")

# 创建复合列 (例如: "Malignant cells_NC")
adata.obs['celltype_group'] = adata.obs['celltype'].astype(str) + "_" + adata.obs['group'].astype(str)

# 挑选需要重点展示机制响应的核心细胞群
focus_celltypes = ['Malignant cells', 'Resident macrophages', 'Activated macrophages',
    'Activated T cells','Exhausted-like T cells','NK cells', 'Endothelial cells']

ordered_categories = []
for ct in focus_celltypes:
    for grp in treatment_order: 
        ordered_categories.append(f"{ct}_{grp}")

# 过滤出存在于数据中的细胞
valid_categories = [cat for cat in ordered_categories if cat in adata.obs['celltype_group'].unique()]

adata_focus = adata[adata.obs['celltype_group'].isin(valid_categories)].copy()
adata_focus.obs['celltype_group'] = pd.Categorical(adata_focus.obs['celltype_group'], categories=valid_categories, ordered=True)

# 精选一些核心双抗相关基因用于展示
show_genes = [g for g in ['Vegfa', 'Kdr', 'Cd274', 'Pdcd1', 'Havcr2', 'Ctla4', 'Lag3', 'Ifng', 'Gzmb'] if g in available_genes]

if len(show_genes) > 0:
    dp = sc.pl.dotplot(
        adata_focus, 
        var_names=show_genes, 
        groupby='celltype_group', 
        standard_scale='var', 
        cmap='Reds',
        return_fig=True
    )
    dp.style(cmap='Reds', dot_edge_color='black', dot_edge_lw=0.5)
    dp.savefig(os.path.join(target_dir, 'Target_Nested_DotPlot.pdf'), dpi=300, bbox_inches='tight')
    print("  Saved: Target_Nested_DotPlot.pdf")
# ==============================================================================
# 6. 联合治疗协同效应分析 (新增模块)
# ==============================================================================
print("\n[Target Analysis] Analyzing combination therapy synergy...")

def calculate_synergy(ak112_val, cd47_val, combo_val, method='Bliss'):
    """
    计算药物协同效应
    Bliss independence: E_combo = E_A + E_B - E_A * E_B
    如果实际效应 > 预期效应 = 协同 (synergy)
    如果实际效应 < 预期效应 = 拮抗 (antagonism)
    """
    if method == 'Bliss':
        expected = ak112_val + cd47_val - (ak112_val * cd47_val)
        synergy_score = combo_val - expected
        return synergy_score, expected
    return None, None

# 示例：对每个基因在每个细胞类型中计算协同
synergy_results = []

for gene in ['Vegfa', 'Cd274', 'Pdcd1', 'Cd47', 'Ifng', 'Gzmb']:
    if gene not in adata.raw.var_names:
        continue

    for celltype in adata.obs['celltype'].unique():
        # 获取各组的平均表达
        nc_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                (df_sample_stats['CellType']==celltype) &
                                (df_sample_stats['Group']=='NC')]['Mean_Expr'].mean()

        ak112_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                    (df_sample_stats['CellType']==celltype) &
                                    (df_sample_stats['Group']=='AK112')]['Mean_Expr'].mean()

        cd47_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                    (df_sample_stats['CellType']==celltype) &
                                    (df_sample_stats['Group']=='CD47')]['Mean_Expr'].mean()

        combo_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                    (df_sample_stats['CellType']==celltype) &
                                    (df_sample_stats['Group']=='AK112+CD47')]['Mean_Expr'].mean()

        if not any(pd.isna([nc_expr, ak112_expr, cd47_expr, combo_expr])):
            # 计算相对于NC的变化（0-1标准化）
            ak112_effect = (ak112_expr - nc_expr) / nc_expr if nc_expr > 0 else 0
            cd47_effect = (cd47_expr - nc_expr) / nc_expr if nc_expr > 0 else 0
            combo_effect = (combo_expr - nc_expr) / nc_expr if nc_expr > 0 else 0

            synergy_score, expected = calculate_synergy(ak112_effect, cd47_effect, combo_effect)

            synergy_results.append({
                'Gene': gene,
                'CellType': celltype,
                'AK112_Effect': ak112_effect,
                'CD47_Effect': cd47_effect,
                'Combo_Effect': combo_effect,
                'Expected_Effect': expected,
                'Synergy_Score': synergy_score,
                'Synergy_Type': 'Synergy' if synergy_score > 0 else 'Antagonism'
            })

df_synergy = pd.DataFrame(synergy_results)
df_synergy.to_csv(os.path.join(target_dir, 'Synergy_Analysis.csv'), index=False)
# 绘制协同效应热图
if not df_synergy.empty:
    plt.figure(figsize=(10, 6))
    synergy_pivot = df_synergy.pivot(index='Gene', columns='CellType', values='Synergy_Score')
    sns.heatmap(synergy_pivot, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
                cbar_kws={'label': 'Synergy Score (>0 = Synergy)'})
    plt.title('AK112 + CD47 Synergy Analysis', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, 'Synergy_Heatmap.pdf'), dpi=300, bbox_inches='tight')
    plt.close()
# ==============================================================================
# 5. 靶点通路级别 Heatmap (矩阵图)
# ==============================================================================
print("\n[Target Analysis] Generating Pathway Heatmap...")
pathway_genes = [g for g in all_target_genes if g in adata.raw.var_names]

if len(pathway_genes) > 0:
    # 按照 Celltype 分组展示整体特征
    sc.pl.matrixplot(adata, var_names=pathway_genes, groupby='celltype', 
                     cmap='RdBu_r', standard_scale='var',
                     save='Target_pathway_heatmap.pdf')
    print("  Saved: Target_pathway_heatmap.pdf")

print("\n" + "="*70)
print("✅ Target Analysis Module Completed Successfully!")
print("="*70)

### VEGF/PD1 Pathway Module Analysis

Module-based analysis of VEGF signaling, PD1 pathway, angiogenesis, and immune exhaustion.


In [ ]:
# ============================================================
# Upgraded VEGF/PD1 Module-based Analysis
# ============================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ----------------------------
# USER CONFIG
# ----------------------------
target_dir = os.path.join(RESULTS_DIR, 'Module_Analysis')
os.makedirs(target_dir, exist_ok=True)

min_cells = 10  # 稍微放宽一点，避免某个样本某些细胞过少被剔除
PLOT_ALL_PANEL = True   
q_cut = 0.05

treat_vs_nc_comps = [('AK112', 'NC'), ('RC148', 'NC')]
drug_vs_drug_comps = [('AK112','RC148')]

# ----------------------------
# 1) Merge celltypes (已修正字典映射)
# ----------------------------
myeloid_types = ['Monocytes', 'Mrc1+ TAMs', 'Ccl8+ TAMs', 'Neutrophils', 'cDCs', 'pDCs', 'Mast cells']
t_types = ['T cells'] # 修正为实际存在的标签

def merge_celltype(ct):
    if ct == 'Malignant cells': return 'Tumor'
    if ct in myeloid_types:     return 'Myeloid'
    if ct == 'NK cells':        return 'NK'
    if ct in t_types:           return 'T'
    if ct in ['Fibroblasts', 'Endothelial cells', 'Pericytes']: return 'Stromal'
    return 'Other'

if 'celltype' in adata.obs.columns:
    adata.obs['celltype_merged'] = adata.obs['celltype'].map(merge_celltype).astype('category')
else:
    adata.obs['celltype_merged'] = 'All'

# ----------------------------
# 2) Unified modules (已拆分配体与受体，符合生物学逻辑)
# ----------------------------
modules = {
    "VEGF_Ligand": {
        "score_genes": ['Vegfa','Vegfb','Vegfc','Pgf'],
        "heatmap_genes": ['Vegfa','Vegfb','Pgf'],
        "celltype": "Tumor",  # 肿瘤是配体主要来源
    },
    "VEGF_Receptor": {
        "score_genes": ['Kdr','Flt1','Flt4','Nrp1','Nrp2'],
        "heatmap_genes": ['Kdr','Flt1','Nrp1'],
        "celltype": "Stromal",  # 主要在内皮细胞等基质中
    },
    "PD1_PDL1_Pathway": {
        "score_genes": ['Cd274','Pdcd1','Pdcd1lg2','Ctla4','Cd28','Cd80','Cd86'],
        "heatmap_genes": ['Cd274','Pdcd1','Pdcd1lg2','Ctla4'],
        "celltype": "T",  
    },
    "Angiogenesis": {
        "score_genes": ['Angpt1','Angpt2','Tek','Pdgfa','Pdgfb','Pdgfra','Pdgfrb'],
        "heatmap_genes": ['Angpt1','Angpt2','Tek','Pdgfa','Pdgfb'],
        "celltype": "Stromal",  
    },
    "Immune_Exhaustion": {
        "score_genes": ['Pdcd1','Ctla4','Lag3','Havcr2','Tigit','Cd244','Eomes','Entpd1','Cd39'],
        "heatmap_genes": ['Pdcd1','Ctla4','Lag3','Havcr2','Tigit'],
        "celltype": "T",  
    },
    "Myeloid_Infiltration": {
        "score_genes": ['Itgam','Cd11b','Ly6c','Ly6g','Cxcr2','Ccr2','Cd14','Fcgr1','Fcgr3'],
        "heatmap_genes": ['Itgam','Cd11b','Ly6c','Cxcr2','Ccr2'],
        "celltype": "Myeloid",
    },
}

# ----------------------------
# 3) Compute module scores (cell-level)
# ----------------------------
for m, cfg in modules.items():
    genes = [g for g in cfg["score_genes"] if g in adata.raw.var_names]
    if len(genes) >= 2: # 放宽到2个基因即可打分
        score_col = f"{m}_score"
        sc.tl.score_genes(adata, gene_list=genes, score_name=score_col, use_raw=True)
        print(f"[score] {m}: {len(genes)}/{len(cfg['score_genes'])} genes used")

# ----------------------------
# 4) Upgraded Violin Plot (Cell-level stats + Sample-level dots)
# ----------------------------
def stars(q):
    if q is None or np.isnan(q): return 'ns'
    if q < 0.001: return '***'
    if q < 0.01:  return '**'
    if q < 0.05:  return '*'
    return 'ns'

def violin_with_fdr_stars(ad_sub, score_col, order, palette, comparisons, title, outpath):
    # Cell-level data for violin and stats
    df_cell = ad_sub.obs[['group', 'sampleID', score_col]].dropna().copy()
    
    # Sample-level data for overlay dots
    df_sample = df_cell.groupby(['group', 'sampleID'])[score_col].mean().reset_index()
    
    pvals = []
    for treat, base in comparisons:
        a = df_cell.loc[df_cell['group'] == treat, score_col].values
        b = df_cell.loc[df_cell['group'] == base, score_col].values
        if len(a) > 5 and len(b) > 5:
            p = mannwhitneyu(a, b, alternative='two-sided')[1]
        else:
            p = np.nan
        pvals.append(p)

    qvals = [np.nan]*len(pvals)
    valid = [i for i,p in enumerate(pvals) if not np.isnan(p)]
    if len(valid) > 0:
        q = multipletests([pvals[i] for i in valid], method='fdr_bh')[1]
        for ii, qi in zip(valid, q):
            qvals[ii] = qi

    plt.figure(figsize=(6, 5))
    # 1. 细胞水平的小提琴图 (展示整体转录偏移)
    ax = sns.violinplot(data=df_cell, x='group', y=score_col, order=order, 
                        palette=palette, inner=None, alpha=0.5, linewidth=0)
    
    # 2. 样本水平的大黑点 (展示生物学重复的真实分布)
    sns.stripplot(data=df_sample, x='group', y=score_col, order=order, 
                  color='black', size=8, jitter=0.1, ax=ax)

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(f'{score_col} (Cell-level)')
    
    # Add stats lines
    ymin, ymax = ax.get_ylim()
    y0 = ymax + (ymax - ymin) * 0.02
    dy = (ymax - ymin) * 0.08
    x_dict = {g:i for i,g in enumerate(order)}
    
    k = 0
    for (g1, g2), q in zip(comparisons, qvals):
        s = stars(q)
        x1, x2 = x_dict[g1], x_dict[g2]
        y = y0 + k * dy
        ax.plot([x1, x1, x2, x2], [y, y+dy*0.2, y+dy*0.2, y], lw=1.2, c='black')
        ax.text((x1+x2)/2, y+dy*0.25, s, ha='center', va='bottom', fontsize=12)
        k += 1
    ax.set_ylim(ymin, y0 + max(1, k+1) * dy)

    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# ----------------------------
# 5) Generate Violins
# ----------------------------
print("\n[Violin] Generating module violins...")
for m, cfg in modules.items():
    score_col = f"{m}_score"
    if score_col not in adata.obs.columns: continue

    for ct in [cfg["celltype"], 'All']:
        ad = adata if ct == 'All' else adata[adata.obs['celltype_merged'] == ct]
        if ad.n_obs < min_cells: continue

        violin_with_fdr_stars(
            ad, score_col, treatment_order, group_colors,
            comparisons=treat_vs_nc_comps,
            title=f"{ct} | {m} (Treat vs NC)",
            outpath=os.path.join(target_dir, f"Violin_{ct}_{m}_TreatVsNC.pdf")
        )

# ----------------------------
# 6) Upgraded Heatmap Helpers (Cell-level stats for delta calculation)
# ----------------------------
def compute_delta_matrix_cell_level(ad_sub, genes, comparison_type):
    # 使用细胞级别的均值差异，并用细胞级数据算p值，保证有统计效力
    deltas, pvals = {}, {}
    base = 'NC' if comparison_type == 'Treatment_vs_NC' else 'AK112'
    comps = ['AK112', 'RC148'] if comparison_type == 'Treatment_vs_NC' else ['RC148']

    expr_df = pd.DataFrame(ad_sub.raw[:, genes].X.toarray(), columns=genes, index=ad_sub.obs_names)
    expr_df['group'] = ad_sub.obs['group'].values

    for treat in comps:
        a = expr_df.loc[expr_df['group'] == treat, genes]
        b = expr_df.loc[expr_df['group'] == base, genes]

        if len(a) > 5 and len(b) > 5:
            deltas[treat] = a.mean(axis=0) - b.mean(axis=0)
            pv = [mannwhitneyu(a[g].values, b[g].values, alternative='two-sided')[1] for g in genes]
            pvals[treat] = pd.Series(pv, index=genes)
        else:
            deltas[treat] = pd.Series([np.nan]*len(genes), index=genes)
            pvals[treat] = pd.Series([np.nan]*len(genes), index=genes)

    delta_df, pval_df = pd.DataFrame(deltas), pd.DataFrame(pvals)
    flat_p = pval_df.values.flatten()
    mask = ~np.isnan(flat_p)
    q = np.full_like(flat_p, np.nan, dtype=float)
    if mask.sum() > 0:
        q[mask] = multipletests(flat_p[mask], method='fdr_bh')[1]
    qval_df = pd.DataFrame(q.reshape(pval_df.shape), index=pval_df.index, columns=pval_df.columns)
    
    return delta_df, pval_df, qval_df

def plot_delta_heatmap(delta_df, title, outpath, qval_df, q_cut):
    # 修复了 applymap 在新版 pandas 中的 deprecation 警告，改用 map
    ann_str = delta_df.apply(lambda col: col.map(lambda x: "" if pd.isna(x) else f"{x:.2f}"))
    sig_mask = (qval_df < q_cut)
    
    for r in ann_str.index:
        for c in ann_str.columns:
            if sig_mask.loc[r, c] and ann_str.loc[r, c] != "":
                ann_str.loc[r, c] = ann_str.loc[r, c] + "*"

    plt.figure(figsize=(max(4, 0.95*delta_df.shape[1] + 3), max(3, 0.50*delta_df.shape[0] + 2)))
    ax = sns.heatmap(delta_df, cmap='vlag', center=0, linewidths=0.5,
                     annot=ann_str, fmt="", cbar_kws={'label': 'Delta (log1p Expr)'})
    ax.set_title(title, pad=15)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# ----------------------------
# 7) Generate Heatmaps
# ----------------------------
print("\n[Heatmap] Generating delta heatmaps...")
for m, cfg in modules.items():
    genes = [g for g in cfg["heatmap_genes"] if g in adata.raw.var_names]
    if len(genes) < 2: continue

    ad = adata[adata.obs['celltype_merged'] == cfg["celltype"]].copy()
    if ad.n_obs < min_cells: continue

    for comp_type in ['Treatment_vs_NC', 'RC148_vs_AK112']:
        delta_df, pval_df, qval_df = compute_delta_matrix_cell_level(ad, genes, comp_type)
        out = os.path.join(target_dir, f"Heatmap_{m}_{cfg['celltype']}_{comp_type}.pdf")
        plot_delta_heatmap(delta_df, f"{m} ({cfg['celltype']}) | {comp_type}", out, qval_df, q_cut)

print("\n✅ Module-based analysis successfully finished!")

---

## Export & Split

### Load Cleaned Data

In [ ]:
adata = sc.read(DATA_DIR+ '/Step1-sce_cleaned.h5ad')
#adata = sc.read(DATA_DIR+ '/Step0-combined_raw_data.h5ad')
adata

### Check Naming

In [ ]:
# Check cell naming
print("\nadata cell names:")
print(adata.obs_names[:5])

### Load Metadata

In [ ]:
meta = pd.read_csv(RESULTS_DIR + "/sce_metadata_all.csv", index_col=0, sep="\t")
meta.head()

### Distribution Check

In [ ]:
ctab = pd.crosstab(meta['celltype'],meta['group'],margins=True)
print(ctab)

### Subset Cells

In [ ]:
# target_cells = meta[meta["celltype"].isin(["Endothelial cells"])].index
cell_types = meta["celltype"].unique().tolist()
target_cells = meta[meta["celltype"].isin(cell_types)].index
sub = adata[
    adata.obs.index.isin(target_cells), :
]  # Subset adata based on these indices
# Assign the 'celltype' column from meta to NKT.obs
sub.obs["celltype"] = meta.loc[sub.obs.index, "celltype"]
sub.obs["group"] = meta.loc[sub.obs.index, "group"]
sub.obs["batch"] = meta.loc[sub.obs.index, "batch"]
# sub.obs["Main_celltype"] = meta.loc[sub.obs.index, "Main_celltype"]
# Save the result
# Print information about the subset
print(f"Subset contains {sub.n_obs} cells and {sub.n_vars} genes")
sub
# sub.write_h5ad(DATA_DIR + "/Step4-sce_hippo_microglia.h5ad", compression="gzip")#

### Gene Info

In [ ]:
# Readgene_info_has.txt文件（假设路径正确，使用TAB分隔）
gene_info = pd.read_csv('/Users/luye/Data/Sig/gene_info_mmu.txt', sep='\t')

# Filter protein_coding，并获取唯一的external_gene_name（假设你的var_names是基因符号，如'MT-ND1'）
protein_coding_genes = gene_info[gene_info['gene_biotype'] == 'protein_coding']['external_gene_name'].dropna().unique().tolist()
print(f"Found {len(protein_coding_genes)} protein-coding genes.")

# 如果你的var_names是ensembl_gene_id，改用：
# protein_coding_genes = gene_info[gene_info['gene_biotype'] == 'protein_coding']['ensembl_gene_id'].dropna().unique().tolist()

# Filter adata，只保留这些基因
sub = sub[:, sub.var_names.isin(protein_coding_genes)]
print(f"After filtering: {sub.shape[1]} genes remaining.")

### Export Matrix

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# Readh5ad文件
#adata = sc.read_h5ad("文件路径.h5ad")

# 导出表达矩阵（稀疏矩阵转为CSR格式）
expr_matrix = sub.X.T
from scipy.io import mmwrite
mmwrite("raw_counts_matrix.mtx", expr_matrix)

### Export Genes

In [ ]:
# Export gene info
pd.DataFrame(sub.var_names).to_csv("genes.csv", index=False, header=False)

### Export Metadata

In [ ]:
# Export cell metadata
sub.obs.to_csv("cell_metadata.csv")

### Export UMAP

In [ ]:
bdata = sc.read(DATA_DIR+ '/Step3-sce_annotated.h5ad')
# Export UMAP coords
umap_coords = pd.DataFrame(bdata.obsm['X_umap_harmony'], index=bdata.obs_names)
umap_coords.columns = ['UMAP_1', 'UMAP_2']
umap_coords.to_csv("umap_coords.csv")

In [ ]:
sc.logging.print_header()